In [1]:
import pandas as pd
import json

In [2]:
# 총 100개의 test 표본에서 NER 생성 결과를 불러옴
results = pd.read_csv('/workspace/datas/generated/flan-t5-base_fewnerd-tuned_fewnerd_small.csv')

results = results[:100]

results.head()

,Sentence,NER,generated_ner
0,In the early 1930s the band moved to the Grill...,"{""building-other"": [""Grill Room""], ""building-h...","{ ""building-hotel"": [""Grill Room of the Taft H..."
1,The final season of minor league play Elkin Me...,"{""location-park"": [""Elkin Memorial Park""]}","{ ""location-park"": [""Elkin Memorial Park""]}"
2,"They finished the season 14–19 , 9–9 in C-USA ...","{""event-sportsevent"": [""C-USA play""]}","{ ""organization-sportsleague"": [""C-USA""]}"
3,"The B-52 pilot , Major Larry G. Messinger , la...","{""product-other"": [""B-52""], ""person-other"": [""...","{ ""product-airplane"": [""B-52""], ""person-soldie..."
4,The Austro-Hungarian Navy built and operated t...,"{""organization-other"": [""Austro-Hungarian Navy""]}","{ ""organization-other"": [""Austro-Hungarian Nav..."


In [4]:
count_dict = dict()

for ner_json in results['NER']:
    json_dict = json.loads(ner_json)
    for entity, values in json_dict.items():
        entity = entity.split('-')[0]
        if entity not in count_dict:
            count_dict[entity] = len(values)
        else:
            count_dict[entity] += len(values)

print(count_dict)

{'building': 7, 'location': 68, 'organization': 62, 'event': 11, 'product': 19, 'person': 48, 'other': 19, 'art': 12}


# flan-t5-large 결과 분석

일단 현재 overfitting하는 경향이 나타났고, weight decay 수치를 높여봐도 오히려 못하는 상황

In [2]:
results = pd.read_csv('/workspace/datas/generated/flan-t5-large-ner-json-conll2003_random-fp32-w1e3-lr4e5-full-promp1-tuned-conll2003.csv')

results.head()

,Sentence,NER,types,generated_ner
0,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...","{""LOC"": [""JAPAN""], ""PER"": [""CHINA""]}",conll2003,"""LOC"": [""JAPAN"", ""CHINA""]"
1,Nadim Ladki,"{""PER"": [""Nadim Ladki""]}",conll2003,"""PER"": [""Nadim Ladki""]"
2,"AL-AIN , United Arab Emirates 1996-12-06","{""LOC"": [""AL-AIN"", ""United Arab Emirates""]}",conll2003,"""LOC"": [""AL-AIN"", ""United Arab Emirates""]"
3,Japan began the defence of their Asian Cup tit...,"{""LOC"": [""Japan"", ""Syria""], ""MISC"": [""Asian Cu...",conll2003,"""LOC"": [""Japan"", ""Syria""], ""MISC"": [""Asian Cup""]"
4,But China saw their luck desert them in the se...,"{""LOC"": [""China"", ""Uzbekistan""]}",conll2003,"""LOC"": [""China"", ""Uzbekistan""]"


In [3]:
def parse_json(json_str):
    try:
        # json 문자열 전처리
        if pd.isna(json_str):
            json_str = ""
        if len(json_str) == 0 or json_str[0] != '{':
            json_str = '{' + json_str + '}'
        
        json_obj = json.loads(json_str)
        return json_obj
    except (json.JSONDecodeError, TypeError):
        return None

In [ ]:
wrong_indices = []
for index, row in results.iterrows():
    ref_json = parse_json(row["NER"])
    pred_json = parse_json(row["generated_ner"])
    
    if str(ref_json) != str(pred_json):
        wrong_indices.append(index)

        print(f"Index {index}: Wrong")

Index 0: Match
Index 12: Match
Index 15: Match
Index 20: Match
Index 25: Match
Index 29: Match
Index 34: Match
Index 39: Match
Index 41: Match
Index 48: Match
Index 49: Match
Index 51: Match
Index 87: Match
Index 145: Match
Index 149: Match
Index 151: Match
Index 152: Match
Index 153: Match
Index 155: Match
Index 157: Match
Index 159: Match
Index 163: Match
Index 165: Match
Index 185: Match
Index 189: Match
Index 199: Match
Index 207: Match
Index 210: Match
Index 212: Match
Index 217: Match
Index 234: Match
Index 235: Match
Index 237: Match
Index 263: Match
Index 305: Match
Index 308: Match
Index 312: Match
Index 314: Match
Index 315: Match
Index 317: Match
Index 334: Match
Index 336: Match
Index 339: Match
Index 341: Match
Index 369: Match
Index 371: Match
Index 374: Match
Index 376: Match
Index 378: Match
Index 379: Match
Index 381: Match
Index 382: Match
Index 390: Match
Index 399: Match
Index 408: Match
Index 426: Match
Index 429: Match
Index 431: Match
Index 432: Match
Index 439: 

In [7]:
wrong_df = results.loc[wrong_indices]

In [9]:
print(len(wrong_df))

543


In [8]:
wrong_df.sample(10)

,Sentence,NER,types,generated_ner
3525,NBA BASKETBALL - FRIDAY 'S RESULTS .,"{""ORG"": [""NBA""]}",conll2003,"""MISC"": [""NBA""]"
955,"Czech ambassador to the United Nations , Karel...","{""LOC"": [""Czech""], ""ORG"": [""United Nations"", ""...",conll2003,"""MISC"": [""Czech"", ""Czechs""], ""PER"": [""Karel Ko..."
1088,Mediterranean oil products were steady to most...,"{""ORG"": [""Elf""]}",conll2003,"""MISC"": [""Mediterranean""], ""ORG"": [""Elf""]"
1897,"Toledo 61,514 0","{""LOC"": [""Toledo""]}",conll2003,"""ORG"": [""Toledo""]"
153,"FIFA 's players ' status committee , meeting i...","{""ORG"": [""FIFA"", ""Udinese""], ""LOC"": [""Barcelon...",conll2003,"""ORG"": [""FIFA""], ""LOC"": [""Barcelona""], ""MISC"":..."
49,"Syria : 24 - Salem Bitar , 3 - Bachar Srour ; ...","{""LOC"": [""Syria""], ""PER"": [""Salem Bitar"", ""Bac...",conll2003,"""LOC"": [""Syria""], ""PER"": [""Salem Bitar"", ""Bach..."
3272,Postponed : Airdrieonians v Clydebank ( to Wed...,"{""ORG"": [""Airdrieonians"", ""Clydebank"", ""East""]}",conll2003,"""ORG"": [""Airdrieonians"", ""Clydebank""], ""LOC"": ..."
871,Bottom team Reggiana are also without a suspen...,"{""ORG"": [""Reggiana""], ""MISC"": [""German""], ""PER...",conll2003,"""ORG"": [""Reggiana""], ""PER"": [""Dietmar Beiersdo..."
1880,U.S. barge rates were lightly quoted Friday on...,"{""LOC"": [""U.S."", ""St. Louis""]}",conll2003,"""LOC"": [""U.S.""], ""ORG"": [""St. Louis Merchants ..."
1071,Trade and Industry Secretary Ian Lang added th...,"{""ORG"": [""Trade and Industry""], ""PER"": [""Ian L...",conll2003,"""PER"": [""Ian Lang""], ""LOC"": [""Britain"", ""Unite..."


# classification inference 결과 살펴보기

In [1]:
import numpy as np
import os
import pandas as pd

In [24]:
result_dir_path = "/workspace/datas/encoder_result/conll2003/flan-t5-base-encoder-switch-ner-custom-class_weight-drop10-smoothing0-cycle10-lr2e-4-cosine_restart"
dataset_csv_path = "/workspace/datas/conll2003/testb.switch.csv"

dataset_df = pd.read_csv(dataset_csv_path)
test_texts = dataset_df['Sentence'].tolist()

labels = np.load(os.path.join(result_dir_path, "test_labels.npy"))
predictions = np.load(os.path.join(result_dir_path, "test_predictions.npy"))

In [14]:
print(predictions.shape)
print(labels.shape)

(14732, 223, 2)
(14732, 223)


In [15]:
predictions_oh = predictions.argmax(axis=2)
print(predictions_oh[1])

[0 0 0 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0]


In [17]:
print(labels[1])

[-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 

In [25]:
valid_predictions = []
valid_labels = []
for cur_row in zip(predictions_oh, labels):
    cur_preds, cur_labels = cur_row
    cur_valid_preds = []
    cur_valid_labels = []
    for p, l in zip(cur_preds, cur_labels):
        if l != -100:
            cur_valid_preds.append(str(p))
            cur_valid_labels.append(str(l))
    valid_predictions.append(' '.join(cur_valid_preds))
    valid_labels.append(' '.join(cur_valid_labels))

dataset_df['true_labels'] = valid_labels
dataset_df['predicted_labels'] = valid_predictions

In [19]:
pd.set_option('display.max_colwidth', None)

In [26]:
dataset_df.head()

,Sentence,NER,true_labels,predicted_labels
0,"Determine whether or not the named entity of type // MISC // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
1,"Determine whether or not the named entity of type // ORG // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
2,"Determine whether or not the named entity of type // PER // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 0 0 0 0 0 1 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
3,"Determine whether or not the named entity of type // LOC // is present in the following sentence // SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .",-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0 1 0 0 0 0 0 0 0 0 0,0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0,0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0
4,Determine whether or not the named entity of type // MISC // is present in the following sentence // Nadim Ladki,-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 0 0,0 0 0 0 0 0 0,0 0 0 0 0 0 0


In [27]:
correct_rows = dataset_df[dataset_df['predicted_labels'] == dataset_df['true_labels']]

print(f"Correct predictions: {len(correct_rows)} / {len(dataset_df)}")

Correct predictions: 13690 / 14732


In [29]:
incorrect_rows = dataset_df[dataset_df['predicted_labels'] != dataset_df['true_labels']]

print(f"Incorrect predictions: {len(incorrect_rows)} / {len(dataset_df)}")

Incorrect predictions: 1042 / 14732


In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

In [47]:
for row in incorrect_rows.sample(10).itertuples():
    sentence = row.Sentence.split("//")[-1].strip()
    entity_type = row.Sentence.split("//")[-3].strip()
    
    sentence_ids = np.array(tokenizer.encode(sentence))
    true_labels = np.array([int(x) for x in row.true_labels.split()])
    predicted_labels = np.array([int(x) for x in row.predicted_labels.split()])
    
    if len(sentence_ids) != len(true_labels) or len(sentence_ids) != len(predicted_labels):
        print("Length mismatch, skipping...")
        print(f"index: {row.Index}")
        continue
    
    label_indices = true_labels == 1
    predicted_indices = predicted_labels == 1
    label_tokens = tokenizer.decode(sentence_ids[label_indices])
    predicted_tokens = tokenizer.decode(sentence_ids[predicted_indices])
    
    print(f"Index: {row.Index}")
    print(f"Entity Type to find: {entity_type}")
    print(f"Sentence: {sentence}")
    print(f"True Labels: {label_tokens}")
    print(f"Predicted Labels: {predicted_tokens}")
    print("-----")

Index: 4696
Entity Type to find: LOC
Sentence: The longest wait to load on the West Coast was 13 days .
True Labels: 
Predicted Labels: West Coast
-----
Index: 6382
Entity Type to find: ORG
Sentence: Wenchang has built a berth for 5,000 deadweight-tonne container ships at the port and invested 34 million yuan ( $ 4.1 million ) to dredge the harbour , Xinhua said .
True Labels: Xinhua
Predicted Labels: Wenchang Xinhua
-----
Index: 9375
Entity Type to find: ORG
Sentence: John Lewis UK store sales up 4.5 % in week .
True Labels: John Lewis UK
Predicted Labels: John Lewis
-----
Index: 14286
Entity Type to find: MISC
Sentence: PACIFIC DIVISION
True Labels: 
Predicted Labels: PACIFIC DIVISION
-----
Index: 9752
Entity Type to find: PER
Sentence: Seagramd ace 20/11/96 5,000 Japan
True Labels: 
Predicted Labels: Seagramd
-----
Index: 5167
Entity Type to find: LOC
Sentence: Baril said that apart from the group of 150,000 , U.S. and British reconnaissance plans had tracked two much smaller groups

# fenerd inference 메모리 문제

16배치 53% -> 1323521~1323536

* 64배치 66% -> 1639424~1639487
* 128배치 66% -> 1639040~1639167
* 192배치 66% -> 1638912~1639103

대략 1639040~1639103 사이에 문제의 장문이 있을 듯

In [8]:
import pandas as pd
import numpy as np

In [2]:
test_set = pd.read_csv("/workspace/datas/fewnerd/supervised/test.switch.csv")

In [ ]:
len_list = list()
for index, row in test_set.iterrows():
    len_list.append(len(row['Sentence'].split()))

print(max(len_list))

318


In [10]:
print(len(test_set) * 0.65)
print(len(test_set) * 0.67)

1615099.2
1664794.56


In [11]:
start_index = 1615099
end_index = 1664794

test_subset = test_set.iloc[start_index:end_index+1]
sub_set_len_list = list()
for index, row in test_subset.iterrows():
    sub_set_len_list.append(len(row['Sentence'].split()))

print(max(sub_set_len_list))

136


In [9]:
np.argmax(len_list)

np.int64(358908)

# 각 데이터셋별 positive 비율

## conll2003

0.05549

## wnut17

0.01374